Apache Kafka is a powerful distributed event streaming system that provides real-time data. Event-driven architectures are the most commonly used strategy for streaming data: continuous generation and transmission of data from sources like sensors, websites, and financial transactions However enterprises often need additional data governance, data security, data integration, observability, and operational data management. Confluent packages all of these capabilities into a data platform that reduces the complexity of deploying and operating Kafka at scale. Running Kafka clusters in production requires managing brokers, storage, networking, scaling, upgrades, monitoring, and disaster recovery. Confluent offers managed cloud services and operational tooling that reduce the burden on infrastructure teams and allow developers to focus on building applications rather than maintaining clusters.

Rather than treating Kafka as a messaging system, Confluent promotes Kafka as a central nervous system for enterprise data. Events can be shared across applications, advanced analytics systems, artificial intelligence and machine learning platforms, and operational systems in real time. This enables data architectures that are more responsive and less dependent on either batch processing or batch data ingestion and that speed data-driven decision-making and business intelligence.

To see how this story plays out, imagine the use cases of a large health insurance provider, “Wellspring Health,” which already runs Apache Kafka on-premises to process healthcare and insurance events.

Their legacy system contains a Kafka environment which handles:

* Medical claims
* Pharmacy transactions
* Fraud detection
* Provider billing events
* Member eligibility updates

The system works but data teams struggle with inconsistent schemas, unknown downstream consumers, custom ETL pipelines, and slow data analytics onboarding. To improve operational efficiency, Wellspring looks to adopt the cloud-based Confluent Cloud in a data modernization initiative to upgrade their streaming platform while still preserving their existing Kafka investments and knowledge.

The highest priorities on the roadmap for Wellspring are to:

1. Detect potentially fraudulent claims in real time
2. Provide access controls to govern sensitive data structures
3. Easily provide Iceberg snapshots for ingestion into a data lake or data warehouse

In this tutorial, you'll see how Wellspring can modernize and optimize their data infrastructure and Kafka architectures with Confluent to easily create derived big data streams, improve data processing, and create dashboards. This will give them a more scalable and unified data infrastructure and analytics ecosystem.

The first step along this path is to take the Kafka streams that they're already using in an on-prem environment and move them to Confluent Cloud. 

You can find the code sections of this tutorial in our [GitHub repository](https://github.com/IBM/ibmdotcom-tutorials) in two different files: confluent_modernization_notebook.ipynb and dashboard_two_stream.py 

# Step 1

Create an account in Confluent Cloud. You can use GitHub, Google, or simply use your own email address. Answer the initial questions however you'd like. When you reach the stage to create your own cluster, select "Explore other cluster types and pricing".

![](./images/sign_up.png)

Create the environment. For the purposes of this demo, call it "default".

![](./images/0_a_create_account.png)

Next, create the cluster. For the purposes of this, this one is named "tutorial_cluster" but you can choose any name that you'd like. Use the Basic cluster for this tutorial since it has pricing compatible with a simple proof of concept.

![](./images/0_b_create_cluster.png)

Now you're ready to create your first Kafka topic. A Kafka topic is the fundamental unit of organization in Apache Kafka. You can think of it as a feed name or logical channel where data records are published and stored.

# Step 2

After launching your cluster you can now create the first topic, called `medical_claims` to store incoming medical claims.

![](./images/1_create_topic.png)

Edit the data contract:

![](./images/2_topic_data_contract.png)

```
{
  "connect.name": "com.wellspring.claims.MedicalClaim",
  "connect.parameters": {
    "io.confluent.connect.avro.field.doc.event_time": "The string is a unicode character sequence.",
    "io.confluent.connect.avro.record.doc": "Schema for a medical procedure claim."
  },
  "doc": "Schema for a medical procedure claim.",
  "fields": [
    {
      "name": "member_id",
      "type": "int"
    },
    {
      "name": "claim_id",
      "type": "int"
    },
    {
      "name": "provider_id",
      "type": "int"
    },
    {
      "name": "procedure_code",
      "type": "string"
    },
    {
      "name": "diagnosis_code",
      "type": "string"
    },
    {
      "name": "claim_amount",
      "type": "float"
    },
    {
      "name": "claim_status",
      "type": "int"
    },
    {
      "doc": "The string is a unicode character sequence.",
      "name": "event_time",
      "type": "string"
    }
  ],
  "name": "MedicalClaim",
  "namespace": "com.wellspring.claims",
  "type": "record"
}
```

Now you'll add the Data Topic to Schema Registry. This provides a single point of Governance via the Schema Registry so that all clients querying topics know what to expect.

Navigate to the Schema Registry by navigating to your cluster:

![](./images/3_d_schema_registry_location.png)

Then select Data Contracts and then Add Data Contract:

![](./images/3_a_schema_registry.png)

Next, enter the data contract for `medical_claims`:

![](./images/3_b_schema_registry.png)

Now select the API endpoint and note the URL. This is how you can access the schema registry from any client.

To see how this works in action, create a user account to access the schema registry remotely.

Select your username from the upper right menu and then navigate the API Keys view view, then then select Add API Key. Enter whatever name you'd like and select `Schema Registry` for the key scope and `default` for the environment.

![](./images/4_add_key.png)

This generates a new key. Download the key, open the downloaded text file, and copy the values into your .env file. Save the API key as `REGISTRY_API_USER`, the API Secret as `REGISTRY_API_SECRET`, and the registry URL as `REGISTRY_URL`. Then save and close your .env file.

You can now test the Schema Registry like so. You'll want to create a Python environment and install the libraries found in the `requirements.txt` file in the project repository.


In [ ]:
%pip install pyiceberg requests confluent-kafka python-dotenv

In [ ]:
import datetime
import json
import os
from random import randint
from time import sleep

import requests
from confluent_kafka import Producer, SerializingProducer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext, StringSerializer
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

In [ ]:
# Load env variables
load_dotenv()

To test that your registry is working correctly, run the following:

In [ ]:
# Setup Registry and Serializer
sr_conf = {
    "url": str(os.getenv("REGISTRY_URL")),
    "basic.auth.user.info": f'{str(os.getenv("REGISTRY_API_USER"))}:{str(os.getenv("REGISTRY_API_SECRET"))}',
}
schema_registry_client = SchemaRegistryClient(sr_conf)

schema_str = schema_registry_client.get_version(
    "medical_claims-value", schema_registry_client.get_versions("medical_claims-value")[-1]
).schema.schema_str

print(schema_str)

schema = json.loads(schema_str)  # type: ignore

print(" ============ Schema Fields ============")
for f in schema["fields"]:
    print(f["name"])

Now you can see how any client access the streams will be able to see the data contract for that stream and any changes that may have occured in it before reading or writing to it.

# Step 3

Now you can create derived topics in Confluent Cloud through an automation that takes the form of a ksqlDB statement. These can check data quality, aggregate or process event streams, send notifications, and many other data workflows. In Confluent architectures, topics are generated through:

* stream processing
* joins
* aggregations
* enrichment pipelines

Often using:

* ksqlDB
* Kafka Streams
* Apache Flink

You'll see how Confleunts ksqlDB can help Wellspring create two derived topics to streamline data access. The first is to track high value claims of more than $10,000.

You'll need a globally scoped key for the ksqlDB operations. Select your username from the upper right menu and then navigate to the API Keys view, then then select Add API Key. Enter whatever name you'd like and select `Global` for the key scope.

 This generates a new key you can use across Confluent Cloud. Download the key and open that text file and copy values into your .env file. Save the `API key` from the text file as `REQUESTS_API_KEY` in your .env and the `API Secret` from the text file as `REQUESTS_API_SECRET` in your text file.

 You'll also copy your User ID from your User Settings. 

 ![](images/4_global_key.png)

Save this to your .env file as `SERVICE_ACCOUNT_ID`.

Open the Environment details in the Environment tab:

![](images/9_environment_id.png)

Copy the env ID and save it to your .env file as `ENV_ID`.

Open the Cluster details in the Cluster tab:

![](images/11_cluster_ID.png)

Save this to your .env file as `CLUSTER_ID`.

First you'll create the ksql database using the REST URL from your environment:

In [ ]:
def create_db():
    payload = {
        "spec": {
            "display_name": "medical-claims-ksqldb",
            "csu": 4,
            "environment": {
                "id": os.getenv("ENV_ID")  # the ID of your environment
            },
            "kafka_cluster": {
                "id": os.getenv("CLUSTER_ID")  # the ID of your cluster
            },
            "credential_identity": {
                "id": os.getenv(
                    "SERVICE_ACCOUNT_ID"
                )  # the ID of your user profile in Confluent Cloud
            },
        }
    }

    response = requests.post(
        "https://api.confluent.cloud/ksqldbcm/v2/clusters",
        json=payload,
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
    )

    print(response.status_code)
    print(response.json())


create_db()

Now you've created a ksql database. This is a purpose-built event streaming database that lets you process and analyze data in Apache Kafka using standard, lightweight SQL syntax. To use your new ksqlDB, go to the Confluent Cloud console and open the ksqlDB menu item. Note that this may take a few minutes to provision and show up in the Confluent Cloud UI.

Go to settings and copy the Endpoint URL.

![](./images/5_ksql_rest_url.png)

Save this to your `.env` file as `KSQL_ENDPOINT`.

Now you can create a base stream to work with ksqlDB:

In [ ]:
def create_base_stream():
    sql = """
        CREATE STREAM medical_claims_stream (
        claim_id VARCHAR,
        member_id VARCHAR,
        provider_id VARCHAR,
        procedure_code VARCHAR,
        diagnosis_code VARCHAR,
        claim_amount DOUBLE,
        claim_status VARCHAR,
        event_time VARCHAR
        )
        WITH (
        KAFKA_TOPIC='medical_claims',
        VALUE_FORMAT='AVRO'
        );
    """

    response = requests.post(
        str(os.getenv("KSQL_ENDPOINT")),
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
        headers={"Content-Type": "application/vnd.ksql.v1+json; charset=utf-8"},
        json={"ksql": sql},
    )

    print(response.status_code)
    if response.status_code != 404:
        print(response.json())


create_base_stream()

Now with that base stream, you can use SQL statements to filter all of the data coming from the base stream into a derived stream for real-time analytics:

In [ ]:
def create_derived_stream():
    sql = """
        CREATE STREAM high_value_claims AS
        SELECT
        claim_id,
        member_id,
        provider_id,
        claim_amount,
        procedure_code
        FROM medical_claims_stream
        WHERE claim_amount > 10000
        EMIT CHANGES;
    """

    response = requests.post(
        str(os.getenv("KSQL_ENDPOINT")),
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
        headers={"Content-Type": "application/vnd.ksql.v1+json; charset=utf-8"},
        json={"ksql": sql},
    )

    print(response.status_code)
    if response.status_code != 404:
        print(response.json())


create_derived_stream()


You can also create complex windowing. For instance, there are certain kinds of procedures that should trigger an audit, and multiple auditable claims in a short period of time indicate that a provider's systems may have been compromised in a data breach.

To help track this, you'll create an `auditor_stream` of claims that are over $5000 for procedures '7371', '2710', and '1831'. Then, create a new stream that captures whether there are multiple items in the auditor stream in any 10 minute period and capture that as `provider_claim_spikes`.

In [ ]:
def create_auditor_stream():
    sql = """
        CREATE STREAM auditor_stream AS
        SELECT
            claim_id,
            member_id,
            provider_id,
            claim_amount,
            procedure_code,
            diagnosis_code
        FROM medical_claims_stream
        WHERE claim_amount > 5000
        AND procedure_code IN ('7371', '2710', '1831')
        EMIT CHANGES;
    """

    response = requests.post(
        str(os.getenv("KSQL_ENDPOINT")),
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
        headers={"Content-Type": "application/vnd.ksql.v1+json; charset=utf-8"},
        json={"ksql": sql},
    )

    print(response.status_code)
    if response.status_code != 404:
        print(response.json())


create_auditor_stream()

We can now create a provider claim spike event stream that will capture when multiple claims for a single provider are submitted in a short window:

In [ ]:
def create_provider_claim_spike():
    sql = """
        CREATE TABLE provider_claim_spikes AS
        SELECT provider_id,
        COUNT(*) AS claim_count,
        SUM(claim_amount) AS total_claim_value
        FROM auditor_stream
        WINDOW TUMBLING (SIZE 10 MINUTES)
        GROUP BY provider_id
        EMIT CHANGES;
    """

    response = requests.post(
        str(os.getenv("KSQL_ENDPOINT")),
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
        headers={"Content-Type": "application/vnd.ksql.v1+json; charset=utf-8"},
        json={"ksql": sql},
    )

    print(response.status_code)
    if response.status_code != 404:
        print(response.json())


create_provider_claim_spike()

One important thing to note is that the creation of these derived streams will append extra characters to the beginning of the name of the derived topic. You can see the correct name in the Topic for your cluster:

![](./images/3_d_topics_view.png)

## Step 4 - Query derived topics

Now that you've created the derived topics, you can query that stream. Run the following code in a new Python session so that you can see results come in as the are published to the base `medical_claims` topic.


In [ ]:
def query_high_value():
    query = """
    SELECT *
    FROM high_value_claims
    EMIT CHANGES;
    """

    response = requests.post(
        str(os.getenv("KSQL_ENDPOINT")) + "/query-stream",
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
        headers={"Content-Type": "application/vnd.ksql.v1+json; charset=utf-8"},
        json={"sql": query},
        stream=True,
    )

    for line in response.iter_lines():
        if line:
            print(line.decode("utf-8"))


def query_provider_spike():
    query = """
    SELECT *
    FROM provider_claim_spikes
    EMIT CHANGES;
    """

    response = requests.post(
        str(os.getenv("KSQL_ENDPOINT")) + "/query-stream",
        auth=HTTPBasicAuth(
            str(os.getenv("REQUESTS_API_KEY")), str(os.getenv("REQUESTS_API_SECRET"))
        ),
        headers={"Content-Type": "application/vnd.ksql.v1+json; charset=utf-8"},
        json={"sql": query},
        stream=True,
    )

    for line in response.iter_lines():
        if line:
            print(line.decode("utf-8"))


Since you haven't created data would trigger a provider spike yet, nothing will show here yet. To create this data, write to the `medical_claims` stream. 

This requires a key in the cluster itself. Navigate to your cluster and then to `API Keys` and create a new key. This key will be scoped to the cluster itself and so can be used as a producer of events. Download this key and copy the `API key`, `API secret` into your `.env` as `SASL_USERNAME` and `SASL_PASSWORD`. Then copy the `Bootstrap server` details as `BOOTSTRAP_SERVERS`.

Now, open a second terminal window and run the following python code. This code block first gets the data contract from the schema registry and uses that to ensure that the message being sent contains all of the correct fields. This data is then sent through the derived streams to demonstrate how data flows through real-time data processing systems.

This creates 20 high value claims which will show up in the base stream and then also in the provider spike stream since it generates 10 high value claims in a short time span for 2 different providers.


In [ ]:
# Setup Registry and Serializer
sr_conf = {
    "url": str(os.getenv("REGISTRY_URL")),
    "basic.auth.user.info": f'{str(os.getenv("REQUESTS_API_KEY"))}:{str(os.getenv("REQUESTS_API_SECRET"))}',
}
schema_registry_client = SchemaRegistryClient(sr_conf)

schema_str = schema_registry_client.get_version(
    "medical_claims-value", schema_registry_client.get_versions("medical_claims-value")[-1]
).schema.schema_str

avro_serializer = AvroSerializer(
    schema_registry_client,  # type: ignore
    schema_str=schema_str,
)  # type: ignore

new_claim = {
    "member_id": 1000,
    "claim_id": 1000,
    "provider_id": 500,
    "procedure_code": "ABCDEF",
    "diagnosis_code": "GHIJKLM",
    "claim_amount": 1100.0,
    "claim_status": 1,
    "event_time": str(datetime.datetime.now()),
}

producer_conf = {
    "bootstrap.servers": os.getenv("BOOTSTRAP_SERVERS"),
    # Required for Confluent Cloud
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    # Confluent Cloud API credentials
    "sasl.username": os.getenv("SASL_USERNAME"),
    "sasl.password": os.getenv("SASL_PASSWORD"),
}

producer = SerializingProducer({**producer_conf, "value.serializer": avro_serializer})

for i in range(20):
    # flagged procedures = '7371', '2710', '1831'

    high_value_claim = {
        "member_id": randint(0, 10000),
        "claim_id": randint(0, 10000),
        "provider_id": randint(0, 10000),
        "procedure_code": "ABCDEF",
        "diagnosis_code": "GHIJKL",
        "claim_amount": 10000.0 + randint(0, 10000),
        "claim_status": 1,
        "event_time": str(datetime.datetime.now()),
    }

    sleep(randint(0, 10))

    producer.produce(topic="medical_claims", value=high_value_claim)

    producer.flush()

To create a provider claim spike, run the following:

In [ ]:
for i in range(20):
    suspicious_claim = {
        "member_id": 1000,
        "claim_id": 1001,
        "provider_id": 502,  # same provider on each claim
        "procedure_code": "7371",  # flagged procedures = '7371', '2710', '1831'
        "diagnosis_code": "GHIJKLM",
        "claim_amount": 6000.0 + (i * 100),
        "claim_status": 1,
        "event_time": str(datetime.datetime.now()),
    }

    producer.produce(topic="medical_claims", value=suspicious_claim)

    producer.flush()


You'll see the terminal window with the derived topics capture the provider spike from the streamed medical claims and print that value to the terminal window.

## Step 5 - Set up Tableflow

Another key offering of Confluent Cloud is Tableflow, which allows you to quickly and easily create Iceberg tables that store metadata and snapshots of stored messages. Iceberg is an open source data store that creates snapshots of real-time data streaming systems. These can be ingested into analytics engines like IBMs watsonx.data or other providers like Snowflake or Amazon S3.

Go to the topics view in Confluent Cloud and select the generated High Value flow. The generated name from the ksql operation will be something like `pksqlc-xxxxxxx-HIGH_VALUE_CLAIMS`. 

![](./images/7_enable_tableflow.png)

Open the topic and click 'Enable Tableflow' in the upper right hand. Select 'Iceberg' and 'Use Confluent Storage'. 

![](./images/8_tableflow.png)

This stores Iceberg snapshots of all events, along with the associated metadata, creating a widely compatible data asset for consumption by any analytics engine or storage in a data center.

## Step 6 - Query Tableflow Iceberg

In order to query the Iceberg datasets create, you'll need to copy the environment ID from the Environment > Details view and the Organization ID from your organizations. Save this to your .env file as `ENVIRONMENT_FOR_ICEBERG`. 

Then copy the Organization ID from your Organization tab:

![](images/10_org_details.png)

Save this your .env file as `ORGANIZATION_FOR_ICEBERG`.

You can test the Iceberg table creation with the following code:

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "confluent",
    type="rest",
    uri=(
        "https://tableflow.us-east-2.aws.confluent.cloud/"
        "iceberg/catalog/"
        f"{os.getenv('ORGANIZATION_FOR_ICEBERG')}/"
        f"{os.getenv('ENVIRONMENT_FOR_ICEBERG')}"
    ),
    credential=(f'{str(os.getenv("REQUESTS_API_KEY"))}:{str(os.getenv("REQUESTS_API_SECRET"))}'),
    header={"X-Iceberg-Access-Delegation": "vended-credentials"},  # pyright: ignore[reportArgumentType]
)

print(catalog.list_namespaces())

ns = catalog.list_namespaces()

ns_name, table_name = catalog.list_tables(ns[0])[0]

table = catalog.load_table(f"{ns_name}.{table_name}")

# 3. Query with filters and column selection
df = table.scan(limit=100).to_pandas()

print(df.head())

This will show any data created in the Iceberg table.

## Step 7 - Dashboard

Finally, you may want to enable Wellspring Health to view a data visualization dashboard for the high value claims and provider claim spike datasets. To do this, you'll create an app using the Streamlit framework that utilizes the data pipelines enabled by Confluent Cloud to provide a dashboard view to stakeholders.

Open the [dashboard_two_stream.py](dashboard_two_stream.py) from the repository to see the Streamlit app. You can run this streamlit app using the following command in your Python environment:

```
streamlit run dashboard_two_stream.py
```

This shows the power of how data streaming platforms like Confluent Cloud can create a modern platform to centralize and simplify Kafka deployments and streamline data collection, validation and data storage. Ensuring that schemas are correctly applied and automatically generating Iceberg tables doesn't require instrumentation or external storage. This all enables high volumes of data to be processed in a low-latency, near real-time fashion that enables real-time insights from a variety of data sources.